# logit-thermo A2-LF：格子の「遠い床」は本物か（GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは 1〜1.5 時間（探索 8.8 の半分の計算）。終わったら自動で切れます。
- ゲート：`docs/A2_LF_GATE.md`。探索 8.8 の格子と同じ 38 本・4 モデルで、差し替え先を「よそ者（ほかの記事の言葉）」と「身内（同じ記事の言葉）」の 2 通りにして、遠い床をくらべます。判定もこのノートブックの中で計算して `A2_explore/lf_summary.json` に書きます。**結果は表示しません。**
- 計算のコード：`scripts/a2_2_collect.py`・`scripts/a2_explore_xy.py`・`scripts/a2_explore_lattice.py` は Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_lf_floor.py`（`3cc503171eaaa70a…`）は埋め込み（データを開く前に固定）。
- **通知**：🚀・モデルごとの ✅・🎉・⚠️。名札は「logit-thermo A2-LF 格子の遠い床（よそ者と身内の差し替え、4 モデル、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-LF 格子の遠い床（よそ者と身内の差し替え、4 モデル、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-LF far floor', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRCXY': '5c5750e76358b86a1840e6894f9cd86c377e6be1f0b9d4104998b5144a1faaea', 'SRCT': '13bb55fe0e3d0a137c7f7269fb27c84a6e6a1e0cfd57ddeeedcbe54de919aed3'})
SRC22, SRCXY, SRCT = _s['SRC22'], _s['SRCXY'], _s['SRCT']
SRCLF = '"""A2-LF（ゲート docs/A2_LF_GATE.md）：格子の「遠い床」は本物か。データを開く前に固定する。GPU（L4）。\nscripts/a2_2_collect.py・scripts/a2_explore_xy.py・scripts/a2_explore_lattice.py を実行したのと同じ名前空間で実行する\n（load_docs・load_model・y_window・Y_PREFIX・token_logp・interaction_map・rep_tokens・sent_index を変えずに使う）。生成はしない。\n\n探索 8.8 の格子と同じ材料（BBC の窓 38 本、本文は先頭から最大 384 トークン、同じ 4 モデル）で、差し替え先を 2 通りにする：\n  よそ者：ほかの記事の本文に出てくるトークン（探索 8.8 と同じ選び方）\n  身内：同じ記事の本文に出てくるトークン（差し替える位置のトークンそのものは除く）。話題も「記事らしさ」も保ったまま、\n        その位置の中身だけを入れ替える。\n差し替える位置 i は 4 つおき（計算を探索 8.8 の半分にするため）。j はすべての位置。\n遠い床 FF ＝ 文の距離 k が 3〜12 の組の Δ の平均。近く S0 ＝ 同じ文（k ＝ 0）の組の Δ の平均。\n分けられるもの：遠い床が「その位置のトークンの中身とのペアごとの結合」（身内に替えても残る）か、「場違いな言葉が入ったことへの\n文章全体の反応」（身内に替えると消える。話題の乱れと記事らしさの乱れは、この実験では分けない）か。\n"""\nimport gc, json, math, time\nimport numpy as np\nimport torch\n\nLF_MODELS = [(\'pythia-1b\', \'EleutherAI/pythia-1b\'), (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\'),\n             (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\'), (\'smollm2-1.7b\', \'HuggingFaceTB/SmolLM2-1.7B\')]\nLF_STRIDE, LF_SEED, LF_BOOT, LF_BOOT_SEED = 4, 20261105, 1000, 20261106\nLF_KFAR = (3, 12)\nLF_TYPES = (\'foreign\', \'same\')\n\n\ndef rep_same(body, rng):\n    """同じ記事の本文のトークンから、出てくる回数に比例して選ぶ。差し替える位置のトークンそのものは選ばない。"""\n    body = np.asarray(body)\n    out = np.empty(len(body), dtype=body.dtype)\n    for i, t in enumerate(body):\n        if np.all(body == t):\n            out[i] = t                        # 全部同じトークンの記事（実際にはない）。そのまま\n            continue\n        while True:\n            c = body[rng.integers(0, len(body))]\n            if c != t:\n                out[i] = c; break\n    return out\n\n\ndef lf_profiles(D, sidx, body):\n    """→ 文の距離 k（0..12）ごとの和と数、遠い床（k ＝ 3〜12）の和と数、そのうち「同じ言葉の繰り返し」（x_j ＝ 元の x_i）の和と数。"""\n    n = D.shape[0]\n    I, J = np.triu_indices(n, 1)\n    v = D[I, J].astype(np.float64)\n    ok = np.isfinite(v)\n    k = sidx[J] - sidx[I]\n    ksum, kcnt = np.zeros(13), np.zeros(13)\n    m = ok & (k <= 12)\n    np.add.at(ksum, k[m], v[m]); np.add.at(kcnt, k[m], 1)\n    far = ok & (k >= LF_KFAR[0]) & (k <= LF_KFAR[1])\n    body = np.asarray(body)\n    rep = far & (body[J] == body[I])\n    return ksum, kcnt, np.array([v[far].sum(), far.sum(), v[rep].sum(), rep.sum()])\n\n\ndef lf_stats(ksum, kcnt, farv, idx=None):\n    """記事ごとの和と数 → FF・S0・繰り返しの割合。idx は記事の選び直し。"""\n    if idx is not None:\n        ksum, kcnt, farv = ksum[idx], kcnt[idx], farv[idx]\n    ks, kc, fv = ksum.sum(0), kcnt.sum(0), farv.sum(0)\n    FF = fv[0] / fv[1] if fv[1] > 0 else float(\'nan\')\n    S0 = ks[0] / kc[0] if kc[0] > 0 else float(\'nan\')\n    share = fv[2] / fv[0] if fv[0] != 0 else float(\'nan\')\n    return FF, S0, share\n\n\ndef lf_readings(per_model):\n    """per_model：{名前: dict(R=..., R_ci=[lo, hi], share_f=..., share_f_ci=[lo, hi])}"""\n    cis = [v[\'R_ci\'] for v in per_model.values()]\n    pts = [v[\'R\'] for v in per_model.values()]\n    if all(np.isfinite(p) and p <= 0.5 and hi < 0.7 for p, (lo, hi) in zip(pts, cis)):\n        q1 = \'遠くの床は場違いな言葉のときだけ出る（背景の場寄り）\'\n    elif all(lo >= 0.7 for lo, hi in cis):\n        q1 = \'遠くの床は身内の言葉でも残る（ペアごとの結合寄り）\'\n    else:\n        q1 = \'はっきりしない\'\n    sh = [v[\'share_f_ci\'] for v in per_model.values()]\n    if all(lo > 0.5 for lo, hi in sh):\n        q2 = \'半分を超える\'\n    elif all(hi < 0.5 for lo, hi in sh):\n        q2 = \'半分より少ない\'\n    else:\n        q2 = \'はっきりしない\'\n    return q1, q2\n\n\ndef lf_model_result(acc):\n    """acc：{型: (ksum, kcnt, farv)}（それぞれ (記事, …)）→ 1 モデルの結果。R ＝ (FF 身内 ÷ FF よそ者) ÷ (S0 身内 ÷ S0 よそ者)。"""\n    out = {}\n    pt = {t: lf_stats(*acc[t]) for t in LF_TYPES}\n    for t in LF_TYPES:\n        out[t] = dict(FF=pt[t][0], S0=pt[t][1], share_repeat=pt[t][2], S_k=(acc[t][0].sum(0) / np.maximum(acc[t][1].sum(0), 1)).tolist())\n    def ratio(p):\n        qf = p[\'same\'][0] / p[\'foreign\'][0] if p[\'foreign\'][0] != 0 else float(\'nan\')\n        qn = p[\'same\'][1] / p[\'foreign\'][1] if p[\'foreign\'][1] != 0 else float(\'nan\')\n        return qf, qn, (qf / qn if qn not in (0, float(\'nan\')) and np.isfinite(qn) else float(\'nan\'))\n    qf, qn, R = ratio(pt)\n    nD = acc[\'foreign\'][0].shape[0]\n    rng = np.random.default_rng(LF_BOOT_SEED)\n    Rb, Sb, FFsb = [], [], []\n    for _ in range(LF_BOOT):\n        idx = rng.integers(0, nD, nD)\n        pb = {t: lf_stats(*acc[t], idx=idx) for t in LF_TYPES}\n        Rb.append(ratio(pb)[2]); Sb.append(pb[\'foreign\'][2]); FFsb.append(pb[\'same\'][0])\n    q = lambda v: [float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))]\n    out.update(Q_far=qf, Q_near=qn, R=R, R_ci=q(Rb), share_f=pt[\'foreign\'][2], share_f_ci=q(Sb), FF_same_ci=q(FFsb))\n    return out\n\n\ndef run_lf(docs, outdir, device, notify=print):\n    wins = [w for w in (y_window(d[\'text\']) for d in docs) if w is not None]\n    texts = [\' \'.join(s for p in w for s in p) for w in wins]\n    sents_all = [[s for p in w for s in p] for w in wins]\n    res = {\'n_docs\': len(wins), \'models\': {}}\n    for name, hf_id in LF_MODELS:\n        t0 = time.time()\n        tok, model = load_model(hf_id, device)\n        pre = tok(Y_PREFIX, add_special_tokens=False).input_ids\n        bos = [tok.bos_token_id] if (tok.bos_token_id is not None and getattr(tok, \'add_bos_token\', False)) else []\n        encs = [tok(t, add_special_tokens=False, return_offsets_mapping=True) for t in texts]\n        bodies = [e[\'input_ids\'][:L_MAXTOK] for e in encs]\n        sidx = [sent_index(t, s, e[\'offset_mapping\'][:L_MAXTOK]) for t, s, e in zip(texts, sents_all, encs)]\n        acc = {t: [np.zeros((len(wins), 13)), np.zeros((len(wins), 13)), np.zeros((len(wins), 4))] for t in LF_TYPES}\n        for j, b in enumerate(bodies):\n            for ti, t in enumerate(LF_TYPES):\n                rng = np.random.default_rng(LF_SEED + 1000 * ti + j)\n                rep = rep_tokens(bodies, j, len(b), rng) if t == \'foreign\' else rep_same(b, rng)\n                D = interaction_map(model, bos + pre + b, len(bos) + len(pre), rep, device, stride=LF_STRIDE)\n                ks, kc, fv = lf_profiles(D, sidx[j], b)\n                acc[t][0][j], acc[t][1][j], acc[t][2][j] = ks, kc, fv\n        r = lf_model_result({t: tuple(acc[t]) for t in LF_TYPES})\n        r[\'seconds\'] = round(time.time() - t0, 1)\n        res[\'models\'][name] = r\n        np.savez_compressed(f\'{outdir}/lf_{name}.npz\', **{f\'{t}_{q}\': acc[t][i] for t in LF_TYPES for i, q in enumerate([\'ksum\', \'kcnt\', \'farv\'])})\n        del model; gc.collect(); torch.cuda.empty_cache()\n        notify(f\'{name} おわり（{time.time() - t0:.0f} 秒）\')\n    res[\'LQ1\'], res[\'LQ2\'] = lf_readings(res[\'models\'])\n    return res\n\n\ndef self_test_lf():\n    rng = np.random.default_rng(0)\n    body = [5, 6, 7, 5, 8]\n    r = rep_same(body, rng)\n    assert all(a != b for a, b in zip(r, body)) and set(r.tolist()) <= set(body)\n    # 文の距離と繰り返しの数え方\n    D = np.full((5, 5), np.nan, np.float32)\n    D[0, 1], D[0, 3], D[0, 4], D[1, 4] = 1.0, 2.0, 4.0, 8.0\n    sidx = np.array([0, 0, 1, 3, 4])\n    ks, kc, fv = lf_profiles(D, sidx, body)\n    # (0,1) k=0, (0,3) k=3（body[3]=5=body[0] → 繰り返し）, (0,4) k=4, (1,4) k=4\n    assert ks[0] == 1 and kc[0] == 1 and ks[3] == 2 and ks[4] == 12 and kc[4] == 2\n    assert fv.tolist() == [14.0, 3.0, 2.0, 1.0]\n    # 判定：身内で遠くだけ消える → 偽物寄り。遠くも近くと同じ割合で弱まる → 本物寄り\n    def acc_of(FF_f, FF_s, S0_f, S0_s, n=30, noise=0.0):\n        out = {}\n        for t, FF, S0 in ((\'foreign\', FF_f, S0_f), (\'same\', FF_s, S0_s)):\n            ks = np.zeros((n, 13)); kc = np.ones((n, 13)); fv = np.zeros((n, 4))\n            ks[:, 0] = S0 + rng.normal(0, noise, n); fv[:, 0] = FF + rng.normal(0, noise * 0.01, n); fv[:, 1] = 1\n            fv[:, 2] = 0.2 * fv[:, 0]; fv[:, 3] = 1\n            out[t] = (ks, kc, fv)\n        return out\n    fake = lf_model_result(acc_of(0.003, 0.0003, 0.6, 0.5, noise=0.01))\n    real = lf_model_result(acc_of(0.003, 0.0025, 0.6, 0.5, noise=0.01))\n    assert lf_readings({m: fake for m in \'abcd\'})[0] == \'遠くの床は場違いな言葉のときだけ出る（背景の場寄り）\', fake\n    assert lf_readings({m: real for m in \'abcd\'})[0] == \'遠くの床は身内の言葉でも残る（ペアごとの結合寄り）\', real\n    assert lf_readings({\'a\': fake, \'b\': real, \'c\': real, \'d\': real})[0] == \'はっきりしない\'\n    assert lf_readings({m: real for m in \'abcd\'})[1] == \'半分より少ない\'\n    return \'OK\'\n'
assert hashlib.sha256(SRCLF.encode()).hexdigest() == '3cc503171eaaa70ac6f6ea4120589c3111224c9f707ddb3a6a6860ee5f6bedd1'
exec(SRC22)
exec(SRCXY)
exec(SRCT)
exec(SRCLF)
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 記事（A2-2 と同じか確かめる）→ 4 モデル × 2 通りの差し替え → 判定 → 保存（結果は表示しない）
import json
try:
    print('自己テスト：', self_test('cuda'), self_test_xy('cuda'), self_test_lattice('cuda'), self_test_lf())
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_explore', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    r = run_lf(docs, '/content/drive/MyDrive/logit-thermo/A2_explore', 'cuda', notify=lambda s: notify('✅ ' + s, tags='white_check_mark'))
    r['a2_lf_floor_sha256'] = '3cc503171eaaa70ac6f6ea4120589c3111224c9f707ddb3a6a6860ee5f6bedd1'
    json.dump(r, open('/content/drive/MyDrive/logit-thermo/A2_explore' + '/lf_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-LFで止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：判定と要約を A2_explore/lf_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
